In [51]:
import torch
import torch.nn as nn 
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset,DataLoader
import nltk
from nltk.tokenize import word_tokenize 

In [52]:
document = """ The Last Train Journey
On a cold winter evening, Daniel arrived at the old, abandoned train station on the outskirts of Blackwood. Snow was beginning to fall, dusting the rusted tracks with a silent, pristine white.
He clutched the tarnished brass pocket watch his grandfather had left him, its steady, rapid ticking the only sound in the freezing air. According to the final letter, the last train would arrive tonight at exactly midnight—a train that was no longer on any modern timetable.
As the wind howled through the skeletal rafters of the platform, a distant, mournful whistle echoed down the valley. A single amber headlight pierced the thick fog. With a screech of ancient iron, a steam locomotive pulled up to the platform, its windows glowing with a warm, amber light. 
The heavy door hissed open. Daniel took a deep breath, stepped aboard, and watched the station vanish into the snowy dark, embarking on a journey into the secrets of his family's past.
Inside the carriage, velvet seats lined the walls and a conductor in a faded uniform stood waiting without a ticket in his hand. He nodded once at Daniel and whispered, "Your grandfather rode this line every winter until the day he disappeared."
The train lurched forward, and the world outside the window began to change. Snow gave way to golden fields, then to a bustling town square that Daniel recognized from old photographs in the attic.
Daniel walked down the narrow aisle and found a compartment where a young man sat reading a letter by candlelight. The man's face was unmistakable—it was his grandfather, decades younger, with the same watch chain glinting at his waistcoat.
Before Daniel could speak, the train slowed again and stopped at a station that should not have existed. A wooden sign read Blackwood Crossing, and beneath it, painted in fading letters, were the words: All debts must be paid.
His grandfather looked up from the letter and met Daniel's eyes with a sorrowful smile. "You should not have come tonight," he said softly, "but since you are here, you must choose whether to leave the past buried or carry its truth into the morning."
Daniel sat across from him as the watch in his pocket slowed its ticking, matching the rhythm of the train's wheels on the iron rails. Outside, the snow began to fall once more, and the amber light in the windows grew brighter, as if the journey itself were remembering every mile it had ever traveled.
When the train finally reached the edge of the valley, Daniel understood at last why the timetable had vanished and why only one passenger was ever meant to board. He folded his grandfather's letter carefully, stepped back onto the platform, and walked home through the dawn with a story the world would never believe and a secret he would finally keep.
"""


In [53]:

nltk.download('punkt')

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\atiku\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\atiku\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [54]:

tokens = word_tokenize(document.lower())

print(tokens)

['the', 'last', 'train', 'journey', 'on', 'a', 'cold', 'winter', 'evening', ',', 'daniel', 'arrived', 'at', 'the', 'old', ',', 'abandoned', 'train', 'station', 'on', 'the', 'outskirts', 'of', 'blackwood', '.', 'snow', 'was', 'beginning', 'to', 'fall', ',', 'dusting', 'the', 'rusted', 'tracks', 'with', 'a', 'silent', ',', 'pristine', 'white', '.', 'he', 'clutched', 'the', 'tarnished', 'brass', 'pocket', 'watch', 'his', 'grandfather', 'had', 'left', 'him', ',', 'its', 'steady', ',', 'rapid', 'ticking', 'the', 'only', 'sound', 'in', 'the', 'freezing', 'air', '.', 'according', 'to', 'the', 'final', 'letter', ',', 'the', 'last', 'train', 'would', 'arrive', 'tonight', 'at', 'exactly', 'midnight—a', 'train', 'that', 'was', 'no', 'longer', 'on', 'any', 'modern', 'timetable', '.', 'as', 'the', 'wind', 'howled', 'through', 'the', 'skeletal', 'rafters', 'of', 'the', 'platform', ',', 'a', 'distant', ',', 'mournful', 'whistle', 'echoed', 'down', 'the', 'valley', '.', 'a', 'single', 'amber', 'headli

In [55]:
Counter(tokens).keys()

# build vocab

vocab = { '<unk>':0}

for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab) 

vocab

{'<unk>': 0,
 'the': 1,
 'last': 2,
 'train': 3,
 'journey': 4,
 'on': 5,
 'a': 6,
 'cold': 7,
 'winter': 8,
 'evening': 9,
 ',': 10,
 'daniel': 11,
 'arrived': 12,
 'at': 13,
 'old': 14,
 'abandoned': 15,
 'station': 16,
 'outskirts': 17,
 'of': 18,
 'blackwood': 19,
 '.': 20,
 'snow': 21,
 'was': 22,
 'beginning': 23,
 'to': 24,
 'fall': 25,
 'dusting': 26,
 'rusted': 27,
 'tracks': 28,
 'with': 29,
 'silent': 30,
 'pristine': 31,
 'white': 32,
 'he': 33,
 'clutched': 34,
 'tarnished': 35,
 'brass': 36,
 'pocket': 37,
 'watch': 38,
 'his': 39,
 'grandfather': 40,
 'had': 41,
 'left': 42,
 'him': 43,
 'its': 44,
 'steady': 45,
 'rapid': 46,
 'ticking': 47,
 'only': 48,
 'sound': 49,
 'in': 50,
 'freezing': 51,
 'air': 52,
 'according': 53,
 'final': 54,
 'letter': 55,
 'would': 56,
 'arrive': 57,
 'tonight': 58,
 'exactly': 59,
 'midnight—a': 60,
 'that': 61,
 'no': 62,
 'longer': 63,
 'any': 64,
 'modern': 65,
 'timetable': 66,
 'as': 67,
 'wind': 68,
 'howled': 69,
 'through': 70,
 

In [56]:
input_sentences = document.split('\n')

In [57]:
def text_to_indices(sentence,vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])

    return numerical_sentence            

In [58]:
input_numerical_sentences = []
for sentence in input_sentences:
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))
input_numerical_sentences

[[1, 2, 3, 4],
 [5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  1,
  14,
  10,
  15,
  3,
  16,
  5,
  1,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  10,
  26,
  1,
  27,
  28,
  29,
  6,
  30,
  10,
  31,
  32,
  20],
 [33,
  34,
  1,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  10,
  44,
  45,
  10,
  46,
  47,
  1,
  48,
  49,
  50,
  1,
  51,
  52,
  20,
  53,
  24,
  1,
  54,
  55,
  10,
  1,
  2,
  3,
  56,
  57,
  58,
  13,
  59,
  60,
  3,
  61,
  22,
  62,
  63,
  5,
  64,
  65,
  66,
  20],
 [67,
  1,
  68,
  69,
  70,
  1,
  71,
  72,
  18,
  1,
  73,
  10,
  6,
  74,
  10,
  75,
  76,
  77,
  78,
  1,
  79,
  20,
  6,
  80,
  81,
  82,
  83,
  1,
  84,
  85,
  20,
  29,
  6,
  86,
  18,
  87,
  88,
  10,
  6,
  89,
  90,
  91,
  92,
  24,
  1,
  73,
  10,
  44,
  93,
  94,
  29,
  6,
  95,
  10,
  81,
  96,
  20],
 [1,
  97,
  98,
  99,
  100,
  20,
  11,
  101,
  6,
  102,
  103,
  10,
  104,
  105,
  10,
  106,
  107,
  1,
  16,
  108,
  109,
  1,

In [59]:

# Training Sequence form
training_sequence = []
for sentence in input_numerical_sentences:
    for i in range(1,len(sentence)):
        training_sequence.append(sentence[:i+1])

training_sequence

[[1, 2],
 [1, 2, 3],
 [1, 2, 3, 4],
 [5, 6],
 [5, 6, 7],
 [5, 6, 7, 8],
 [5, 6, 7, 8, 9],
 [5, 6, 7, 8, 9, 10],
 [5, 6, 7, 8, 9, 10, 11],
 [5, 6, 7, 8, 9, 10, 11, 12],
 [5, 6, 7, 8, 9, 10, 11, 12, 13],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3, 16],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3, 16, 5],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3, 16, 5, 1],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3, 16, 5, 1, 17],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3, 16, 5, 1, 17, 18],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3, 16, 5, 1, 17, 18, 19],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 10, 15, 3, 16, 5, 1, 17, 18, 19, 20],
 [5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  1,
  14,
  10,
  15,
  3,
  16,
  5,
  1,
  17,
  18,
  19

In [60]:
len_list =[]
for sequence in training_sequence:
    len_list.append(len(sequence))

In [61]:

padded_training_sequence = []

for sequence in training_sequence:
    padded_training_sequence.append([0] * (max(len_list) - len(sequence)) + sequence)



In [62]:
import torch
padded_training_sequence = torch.tensor(padded_training_sequence,dtype=torch.long)
X = padded_training_sequence[:,:-1]
y = padded_training_sequence[:,-1]

X


tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0, 245,  ..., 265,  33,  56],
        [  0, 245,   1,  ...,  33,  56, 246],
        [245,   1,   3,  ...,  56, 246, 266]])

In [63]:
y

tensor([  2,   3,   4,   6,   7,   8,   9,  10,  11,  12,  13,   1,  14,  10,
         15,   3,  16,   5,   1,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         10,  26,   1,  27,  28,  29,   6,  30,  10,  31,  32,  20,  34,   1,
         35,  36,  37,  38,  39,  40,  41,  42,  43,  10,  44,  45,  10,  46,
         47,   1,  48,  49,  50,   1,  51,  52,  20,  53,  24,   1,  54,  55,
         10,   1,   2,   3,  56,  57,  58,  13,  59,  60,   3,  61,  22,  62,
         63,   5,  64,  65,  66,  20,   1,  68,  69,  70,   1,  71,  72,  18,
          1,  73,  10,   6,  74,  10,  75,  76,  77,  78,   1,  79,  20,   6,
         80,  81,  82,  83,   1,  84,  85,  20,  29,   6,  86,  18,  87,  88,
         10,   6,  89,  90,  91,  92,  24,   1,  73,  10,  44,  93,  94,  29,
          6,  95,  10,  81,  96,  20,  97,  98,  99, 100,  20,  11, 101,   6,
        102, 103,  10, 104, 105,  10, 106, 107,   1,  16, 108, 109,   1, 110,
        111,  10, 112,   5,   6,   4, 109,   1, 113,  18,  39, 1

In [64]:
class CustomDataset(Dataset):
    def __init__(self,X,y):
        self.X = X 
        self.y = y
    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]

In [65]:
dataset = CustomDataset(X,y)
len(dataset)

536

In [66]:
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)

In [67]:
for input,output in dataloader:
    print(len(input))
    print(input,output)

32
tensor([[  0,   0,   0,  ...,  33, 217, 218],
        [  0,   0,   0,  ...,  11, 184, 185],
        [  0,   0,   0,  ...,  10,  81,  96],
        ...,
        [  0,   0,   0,  ..., 213, 214,  20],
        [  0,   0,   0,  ...,  55,  10,   1],
        [  0,   0,   0,  ...,  53,  24,   1]]) tensor([ 10,  10,  20,  40, 181,  24, 236,  20,  87, 146, 154,  44,  55,  40,
          1,  29, 147, 112, 171,  18, 241,  10, 187,  13,  21, 109,  33, 252,
        258, 134,   2,  54])
32
tensor([[  0,   0,   0,  ...,   5,   6,   7],
        [  0,   0,   0,  ...,  58,  13,  59],
        [  0,   0,   0,  ...,  70,   1, 261],
        ...,
        [  0,   0,   0,  ...,   1, 113,  18],
        [  0,   0,   0,  ...,  92, 160,   1],
        [  0,   0,   0,  ...,  29,   6,  95]]) tensor([  8,  60,  29,   1,  50,  33, 207,   1, 246,   1,  55, 195,  78, 256,
         11, 137, 189,  10,  20,  39,  33,  41, 204, 179, 104,  10, 147,   1,
        215,  39,  55,  10])
32
tensor([[  0,   0,   0,  ..., 238,  10,  

## LSTM Cell

In [68]:
class LSTMModel(nn.Module):
    def __init__(self,vocab_size):
        super(LSTMModel,self).__init__()
        self.embedding = nn.Embedding(vocab_size,100)
        self.lstm = nn.LSTM(100,150,batch_first=True)
        self.fc = nn.Linear(150,vocab_size)

    def forward(self,x):
        embedding = self.embedding(x)
        intermediate_hidden_state, (final_hidden_state,cell_state) = self.lstm(embedding)
        output = self.fc(final_hidden_state.squeeze(0))
        return output

In [69]:
model = LSTMModel(len(vocab))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model.to(device)

cuda


LSTMModel(
  (embedding): Embedding(267, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=267, bias=True)
)

In [70]:

#^ Training Loop
epochs = 50
learning_rate = .001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=learning_rate)

for epoch in range(epochs):
    total_loss = 0
    
    for batch_x,batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output,batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch: {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}")    
    


Epoch: 1/50, Loss: 5.56996805527631
Epoch: 2/50, Loss: 5.211426005643957
Epoch: 3/50, Loss: 4.848343119901769
Epoch: 4/50, Loss: 4.579690372242647
Epoch: 5/50, Loss: 4.290884326486027
Epoch: 6/50, Loss: 3.9922504705541275
Epoch: 7/50, Loss: 3.704118490219116
Epoch: 8/50, Loss: 3.4065424414242016
Epoch: 9/50, Loss: 3.1070564774905933
Epoch: 10/50, Loss: 2.8212586991927204
Epoch: 11/50, Loss: 2.5451237874872543
Epoch: 12/50, Loss: 2.2684644600924324
Epoch: 13/50, Loss: 2.0328345088397755
Epoch: 14/50, Loss: 1.7986436310936422
Epoch: 15/50, Loss: 1.588229452862459
Epoch: 16/50, Loss: 1.3924440145492554
Epoch: 17/50, Loss: 1.2266385064405554
Epoch: 18/50, Loss: 1.069054901599884
Epoch: 19/50, Loss: 0.94157975210863
Epoch: 20/50, Loss: 0.8244582695119521
Epoch: 21/50, Loss: 0.7194287180900574
Epoch: 22/50, Loss: 0.6381296164849225
Epoch: 23/50, Loss: 0.5632000688244315
Epoch: 24/50, Loss: 0.5007130328346702
Epoch: 25/50, Loss: 0.44642628466381745
Epoch: 26/50, Loss: 0.39956195389523225
Epoc

In [71]:

#^ Prediction
def prediction(model, vocab, text):
    tokenized_text = word_tokenize(text.lower())
    text_indices = text_to_indices(tokenized_text, vocab)
    padded_text = torch.tensor([0] * (max(len_list) - len(text_indices)) + text_indices, dtype=torch.long).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(padded_text)
    
    value, index = torch.max(output, dim=1)
    predicted_token_index = index.item()
    predicted_token = list(vocab.keys())[predicted_token_index]
    
    return text + " " + predicted_token


In [72]:

text = "On a cold winter evening, daniel arrived at the old,"
prediction(model, vocab, text)


'On a cold winter evening, daniel arrived at the old, abandoned'

In [ ]:
import time
#^ Generate automatically - like GPT token generation

num_tokens = 20
input_text = "On a cold winter evening, daniel arrived at the old,"

for i in range(num_tokens):
    # Get prediction
    predicted_full = prediction(model, vocab, input_text)
    
    # Extract only the newly predicted token (last word)
    new_token = predicted_full.split()[-1]
    
    # Append to accumulated text
    input_text += " " + new_token
    
    print(f"{input_text}")
    time.sleep(0.3)  # Add a delay between predictions


Starting text: On a cold winter evening, daniel arrived at the old,

On a cold winter evening, daniel arrived at the old, abandoned
On a cold winter evening, daniel arrived at the old, abandoned train
On a cold winter evening, daniel arrived at the old, abandoned train station
On a cold winter evening, daniel arrived at the old, abandoned train station on
On a cold winter evening, daniel arrived at the old, abandoned train station on the
On a cold winter evening, daniel arrived at the old, abandoned train station on the outskirts
On a cold winter evening, daniel arrived at the old, abandoned train station on the outskirts of
On a cold winter evening, daniel arrived at the old, abandoned train station on the outskirts of blackwood
On a cold winter evening, daniel arrived at the old, abandoned train station on the outskirts of blackwood .
On a cold winter evening, daniel arrived at the old, abandoned train station on the outskirts of blackwood . snow
On a cold winter evening, daniel arri